# Lesson 1: Python Fundamentals

**Week 3 · Data Engineering Course**

---

SQL runs in pgAdmin. Python runs here — in a Jupyter notebook. Each grey box is a **code cell**. Press **Shift+Enter** to run it and see the output underneath.

This lesson teaches the Python you will use every day as a data engineer: how to store values, manipulate text, organise data, make decisions, and write reusable functions.

Every example in this lesson uses sales data — the same kind of data you cleaned in SQL during Weeks 1 and 2.

---

## 1.1 Variables and Data Types

A **variable** stores a value. You name it, assign a value with `=`, and use it later.

Python has five core types you will use constantly:

| Type | Example | What it holds |
|------|---------|---------------|
| `int` | `42` | Whole numbers |
| `float` | `149.99` | Decimal numbers |
| `str` | `'Alice'` | Text |
| `bool` | `True` | Yes or No |
| `None` | `None` | Nothing / missing |

In [ ]:
order_id   = 1001
price      = 149.99
customer   = 'Alice Johnson'
is_paid    = True
refund_date = None

print(order_id)
print(price)
print(customer)
print(is_paid)
print(refund_date)

In [ ]:
# type() tells you what kind of value a variable holds
print(type(order_id))    # int
print(type(price))       # float
print(type(customer))    # str
print(type(is_paid))     # bool
print(type(refund_date)) # NoneType

---

## 1.2 String Operations

In data engineering, most of your time is spent fixing messy text: removing extra spaces, fixing casing, stripping currency symbols.

Python strings have built-in methods for all of this. A method is called with a dot: `value.method()`.

In [ ]:
# Real problem: categories in the CSV have inconsistent casing and whitespace
raw = '  ELECTRONICS  '

print(raw.strip())           # remove leading and trailing spaces
print(raw.strip().lower())   # lowercase
print(raw.strip().title())   # Title Case — the format we want

# You can chain methods: each one returns a string, and the next runs on that
cleaned = raw.strip().title()
print(cleaned)  # 'Electronics'

In [ ]:
# Real problem: price is stored as '$1,200.00' but we need the number 1200.0
price_str = '$1,200.00'

# Step 1: remove the $ sign
step1 = price_str.replace('$', '')
print(step1)   # '1,200.00'

# Step 2: remove the comma
step2 = step1.replace(',', '')
print(step2)   # '1200.00'

# Step 3: convert the string to a float
price = float(step2)
print(price)   # 1200.0
print(type(price))  # float

# All in one line:
price = float(price_str.replace('$', '').replace(',', ''))
print(price)

In [ ]:
# f-strings: embed values inside text
product  = 'Laptop'
quantity = 3
unit_price = 1200.00
total = quantity * unit_price

print(f'Product: {product}')
print(f'Quantity: {quantity}')
print(f'Total: ${total}')

# Format numbers with :.2f (2 decimal places) and :, (thousands separator)
print(f'Total: ${total:,.2f}')   # $3,600.00

---

## 1.3 Lists

A **list** holds multiple values in order. In data engineering, a list often holds a collection of rows, values from a column, or file names to process.

In [ ]:
# A list of regions
regions = ['North', 'South', 'East', 'West']

print(regions)        # the whole list
print(regions[0])     # first item (counting starts at 0)
print(regions[-1])    # last item
print(len(regions))   # number of items: 4

In [ ]:
# Adding to a list
regions.append('Central')
print(regions)   # ['North', 'South', 'East', 'West', 'Central']

# Looping over a list
for region in regions:
    print(f'Region: {region}')

In [ ]:
# List comprehension: a compact way to build a new list from an existing one
# Format: [expression for item in list]

raw_categories = ['ELECTRONICS', '  Clothing  ', 'HOME & KITCHEN']

cleaned = [c.strip().title() for c in raw_categories]
print(cleaned)  # ['Electronics', 'Clothing', 'Home & Kitchen']

---

## 1.4 Dictionaries

A **dictionary** maps keys to values. Each row of a CSV file becomes a dictionary when you read it with Python — the column names are the keys.

In [ ]:
# A single sales row as a dictionary
row = {
    'order_id':   1001,
    'customer':   'Alice Johnson',
    'product':    'Laptop',
    'quantity':   2,
    'unit_price': 1200.00
}

# Access values by key
print(row['customer'])   # Alice Johnson
print(row['quantity'])   # 2

# Add a new key
row['total'] = row['quantity'] * row['unit_price']
print(row['total'])   # 2400.0

In [ ]:
# Loop over all keys and values
for key, value in row.items():
    print(f'  {key}: {value}')

In [ ]:
# .get() is safer than [] when a key might be missing
# [] raises a KeyError if the key does not exist
# .get() returns None (or a default you choose)

print(row.get('region'))           # None — key does not exist, no error
print(row.get('region', 'Unknown'))  # 'Unknown' — custom default

---

## 1.5 Control Flow

`if` / `elif` / `else` lets your code make decisions. `for` loops repeat an action for each item in a sequence.

In [ ]:
# Classify a row's total revenue
total = 1200.00

if total >= 500:
    tier = 'High'
elif total >= 100:
    tier = 'Medium'
else:
    tier = 'Low'

print(f'${total:.2f} is a {tier} order')

In [ ]:
# Loop with enumerate() to get the index alongside the value
files = ['sales_q1.csv', 'sales_q2.csv', 'sales_q3.csv']

for i, filename in enumerate(files, start=1):
    print(f'File {i}: {filename}')

In [ ]:
# Validate a list of rows and count problems
rows = [
    {'order_id': '1001', 'quantity': '2',  'unit_price': '1200.00'},
    {'order_id': '1002', 'quantity': '-1', 'unit_price': '35.00'},   # bad quantity
    {'order_id': '1003', 'quantity': '0',  'unit_price': '75.00'},   # bad quantity
    {'order_id': '1004', 'quantity': '1',  'unit_price': '400.00'},
]

problems = 0
for row in rows:
    qty = int(row['quantity'])
    if qty <= 0:
        print(f'Order {row["order_id"]}: invalid quantity ({qty})')
        problems += 1

print(f'\n{problems} problem rows found')

---

## 1.6 Functions

A function is a named, reusable block of code. In a data pipeline, every cleaning step becomes a function — you write the logic once and call it for every row across every file.

**Rule of thumb:** if you are writing the same code more than twice, put it in a function.

In [ ]:
# A basic function
def clean_price(value):
    '''Convert a price string like $1,200.00 to a float.'''
    cleaned = str(value).replace('$', '').replace(',', '').strip()
    return float(cleaned)

print(clean_price('$1,200.00'))  # 1200.0
print(clean_price('350.00'))     # 350.0
print(clean_price('  $75  '))    # 75.0

In [ ]:
# The problem: clean_price('N/A') crashes with a ValueError
# In a pipeline, one bad row should not crash the whole run
# Solution: use try/except to handle the error gracefully

def clean_price(value):
    '''Convert a price string to float. Returns None if the value cannot be parsed.'''
    try:
        cleaned = str(value).replace('$', '').replace(',', '').strip()
        return float(cleaned)
    except ValueError:
        return None  # return None instead of crashing

print(clean_price('$1,200.00'))  # 1200.0
print(clean_price('N/A'))        # None — safe
print(clean_price(''))           # None — safe

In [ ]:
# Default parameter values: make parameters optional
def clean_text(value, case='title'):
    '''Strip whitespace and normalise the case of a string.

    case can be 'title', 'lower', or 'upper'. Default is 'title'.
    '''
    cleaned = str(value).strip()
    if case == 'lower':
        return cleaned.lower()
    elif case == 'upper':
        return cleaned.upper()
    return cleaned.title()   # default

print(clean_text('  ELECTRONICS  '))            # 'Electronics'
print(clean_text('  electronics  ', 'lower'))   # 'electronics'
print(clean_text('  home & kitchen  '))         # 'Home & Kitchen'

In [ ]:
# A function can return multiple values
def validate_row(row):
    '''Check a row for common data problems.
    Returns (is_valid, reason).
    '''
    try:
        qty = float(row.get('quantity', 0))
    except ValueError:
        return False, 'quantity is not a number'

    if qty <= 0:
        return False, f'quantity is {qty} (must be > 0)'

    return True, 'OK'

test_rows = [
    {'order_id': '1001', 'quantity': '2'},
    {'order_id': '1002', 'quantity': '-1'},
    {'order_id': '1003', 'quantity': 'abc'},
]

for row in test_rows:
    valid, reason = validate_row(row)
    status = 'OK' if valid else 'SKIP'
    print(f'Order {row["order_id"]}: [{status}] {reason}')

In [ ]:
# Putting it together: apply cleaning functions to a list of rows

raw_rows = [
    {'order_id': '1001', 'category': '  ELECTRONICS  ', 'unit_price': '$1,200.00', 'quantity': '2'},
    {'order_id': '1002', 'category': 'clothing',         'unit_price': '35.00',     'quantity': '-1'},
    {'order_id': '1003', 'category': 'HOME & KITCHEN',   'unit_price': '$80.00',    'quantity': '1'},
]

cleaned_rows = []
for row in raw_rows:
    # Clean fields
    row['category']   = clean_text(row['category'])
    row['unit_price'] = clean_price(row['unit_price'])

    # Validate
    valid, reason = validate_row(row)
    if not valid:
        print(f'Skipping order {row["order_id"]}: {reason}')
        continue

    cleaned_rows.append(row)

print(f'\nClean rows: {len(cleaned_rows)}')
for r in cleaned_rows:
    print(r)

---

## Key Takeaways

1. **Variables** store values. Python figures out the type automatically.
2. **String methods** like `.strip()`, `.lower()`, `.title()`, `.replace()` are your main tools for cleaning text. They can be chained.
3. **f-strings** (`f'Total: {total:.2f}'`) are the clearest way to format output.
4. **Lists** hold ordered collections. Loop over them with `for item in list`.
5. **Dictionaries** map keys to values. Use `.get(key, default)` when a key might be missing.
6. **Functions** make code reusable. Write one function per task.
7. **`try/except`** prevents one bad row from crashing the whole pipeline. Return `None` from a cleaning function when the value cannot be fixed — don't raise an error.